# Introduction to the Model Context Protocol
In this module, we study the Model Context Protocol (MCP), an open protocol that connects agent applications to external tools, data, and prompt templates through a single message format. This lecture introduces the integration problem MCP solves, the host/client/server architecture, the three server primitives, the two transports, and the open-source ecosystem that maintains the protocol.

> __Learning Objectives__
>
> By the end of this lecture, you will be able to:
> * __State the integration problem:__ Explain why connecting host applications to tool sources required a custom integration for each pair before MCP, and name the host, client, and server roles the protocol defines.
> * __Distinguish primitives and transports:__ Identify the three server primitives (tools, resources, and prompts) and the two transports (stdio and Streamable HTTP), and state which of them this module implements.
> * __Describe the ecosystem:__ Describe the open-source MCP ecosystem, including the specification's governance, the official SDKs, and the reference servers.

To see why a shared protocol is needed at all, we start with the problem it replaces. Let's get started!
___

## The Tool-Integration Problem
A large language model generates text. For an application to act on that text, e.g., query a database, read a file, or run a calculation, the application must connect the model to external tools. Before MCP, each host application integrated each tool source separately: every pairing had its own message format and its own custom code to write and maintain. Connecting $M$ host applications to $N$ tool sources required $M\cdot N$ custom integrations.

MCP replaces this with one protocol. Each host application implements MCP once, each tool source implements MCP once, and any host can then talk to any server. The $M\cdot N$ integration problem becomes $M + N$ implementations.

The protocol defines three roles:

> __Host__
>
> The application the user interacts with, e.g., a chat interface, a coding assistant, or an agent framework. The host owns the language model, decides which servers to connect to, and mediates every interaction between the model and the servers.

> __Client__
>
> The protocol component inside the host that manages the connection to a single server. The host creates one client per server; the client sends the requests, receives the responses, and tracks the session state.

> __Server__
>
> A separate program that exposes capabilities (tools, resources, and prompts) over the protocol. A server does not know which host or model is on the other side; it only speaks MCP.

These three roles fit together in a fixed arrangement. Let's look at how a session is organized.
___

## Architecture
The host process owns one client for each server it connects to. Each client holds a single session with its server: the pair completes an initialization handshake, and the client then issues requests that the server answers. Servers expose their capabilities through the protocol itself, so the host discovers at runtime what each server offers.

The language model sits on the host side of this boundary. In a production deployment, the host sends the servers' tool descriptors to the model as part of its context; the model selects a tool and constructs arguments for it; the host executes the call through the client and returns the result to the model, which uses it to continue. The model never talks to a server directly: every call passes through the host.

This module has no LLM. You play the host: you read the tool descriptors, select a tool, construct the arguments, make the call, and interpret the result. Every message you send is the same message a production host would send.

What exactly can a server expose? The protocol defines three kinds of capability.
___

## Server Primitives
A server exposes its capabilities through three primitives, distinguished by who controls when each is used:

> __Tools__
>
> Model-controlled actions. A tool is a function the model can choose to call, described by a name, a description, and a JSON Schema for its arguments. Tools are the focus of this module.

> __Resources__
>
> Application-controlled data. A resource is content the host application can read and attach to the model's context, such as a file or a database record. The application, not the model, decides when to use it.

> __Prompts__
>
> User-controlled templates. A prompt is a reusable message template the user invokes explicitly, such as a slash command in a chat interface.

Only tools are implemented in this module's server. Limiting the implementation to tools keeps the server small enough to read end to end, and tool calling carries the module's central idea: MCP is how you expose your own computational code to an agent.

Primitives define what a server offers; transports define how the messages travel.
___

## Transports
MCP messages are independent of the channel that carries them. The specification defines two standard transports:

> __stdio__
>
> The server runs as a local subprocess launched by the host. The client writes JSON-RPC messages to the server's standard input and reads responses from its standard output, one message per line. Standard output is reserved for protocol messages; logs go to standard error. This module implements the stdio transport.

> __Streamable HTTP__
>
> The server runs as an independent service, local or remote, reachable over HTTP. This transport serves deployments where the server cannot be a subprocess of the host. We describe it here but do not implement it.

This module completes the course's networking arc. Module 1 worked with files and JSON, the format every MCP message is written in. Module 2 consumed REST web services: stateless request/response, where each exchange stands alone. Module 3 held a persistent WebSocket connection that either side could push messages over. MCP runs stateful JSON-RPC sessions over transports like these: a session opens with a handshake, both sides carry session state, and messages flow until the session ends.
___

## The Open-Source Ecosystem
MCP is an open specification with public implementations on both sides of the protocol. Anthropic released the specification in November 2024 and donated it to the Agentic AI Foundation within the Linux Foundation in December 2025, placing the protocol under open governance. Official SDKs implement the protocol in several languages, including TypeScript and Python. The reference servers (Everything, Filesystem, Fetch, and Time), published at [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers), are readable examples of complete servers, and the specification is documented at [modelcontextprotocol.io](https://modelcontextprotocol.io).

This module's server implements the same wire protocol in Julia at teaching scale: a tools-only subset small enough to read in full, speaking the same messages as the servers above.
___

## Security Considerations
Connecting a model to executable tools has security consequences, and the protocol's design assigns responsibility for them to the host:

* **Servers execute code on the user's machine.** A stdio server is a subprocess running with the user's permissions; installing a server means trusting the code it runs.
* **Hosts require explicit user consent for tool calls.** The host, not the model, decides whether a call proceeds.
* **Tool descriptions are untrusted input to the model.** Descriptions come from the server and enter the model's context, so the host should treat them with the same caution as any other external content.
___

## Looking Ahead
The next lecture takes the protocol apart message by message: the JSON-RPC 2.0 message shapes, the session lifecycle, tool discovery and invocation, and the two failure modes, all illustrated with messages captured from this module's server. After that, the demo notebook drives a complete live session, and the activities put you on both sides of the protocol: first driving the client, then extending the server with a tool of your own.
___

## Summary
This lecture introduced the Model Context Protocol: the integration problem it solves, the host/client/server architecture, the three server primitives, the two transports, and the ecosystem that maintains the specification.

> __Key Takeaways:__
>
> * **One protocol replaces per-pair integrations:** Connecting many host applications to many tool sources once required a separate integration for every host-source pair, a count that grows with their product; with MCP, each side implements the protocol once, so the total grows only with their sum.
> * **The host owns the model:** A host runs one client per server, sends tool descriptors to the model, and executes the model's tool selections through the client; servers expose capabilities without knowing who is calling.
> * **Open governance, readable implementation:** The specification, released by Anthropic in November 2024 and donated to the Agentic AI Foundation within the Linux Foundation in December 2025, is implemented by official SDKs and reference servers; this module implements the same protocol in Julia at teaching scale.

The next lecture examines the messages themselves, one captured exchange at a time.
___